# Neural Network Training Notebook for Fraud Detection

## Overview
This notebook provides a complete workflow for training a neural network model for fraud detection using the `nntab` package. The model is designed to work with tabular data and includes data preprocessing, model initialization, training, and evaluation steps.

## Prerequisites
Before running this notebook, ensure you have:
- Python 3.7 or higher
- PyTorch installed
- The `nntab` package accessible in your Python path
- Access to your training dataset (CSV or Parquet format)
- A features configuration file (CSV format)
- GPU support (optional but recommended for faster training)

## Quick Start Guide
1. **Setup Environment**: Import all required packages (cells 1-5)
2. **Load Data**: Configure data paths and load training/validation data (cells 6-9)
3. **Initialize Model**: Set up the neural network architecture (cell 10)
4. **Train Model**: Execute the training loop (cell 11)

## Important Notes
- Adjust file paths according to your local system configuration
- Modify hyperparameters (epochs, batch_size, hidden layers) based on your dataset size and computational resources
- The training process will save checkpoints and logs with the experiment name you specify

### import general packages

## Step 1: Import General Python Packages

**Purpose**: Import essential Python libraries for data manipulation, numerical operations, and deep learning.

**What these packages do**:
- `os`: Operating system interface for file/directory operations
- `pandas`: Data manipulation and analysis library
- `numpy`: Numerical computing library for array operations
- `torch`: PyTorch deep learning framework

**Action Required**: Run the following cells to import general packages.

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
import torch

### import nntab

## Step 2: Import the nntab Package

**Purpose**: Import the custom `nntab` package which contains all the necessary modules for neural network training on tabular data.

**Important Configuration**:
- The `sys.path.append()` command adds the nntab package location to your Python path
- **You MUST modify** the path in the next cell to point to where your `nntab` package is on your system

**Package Modules**:
- `datasets`: Data loading, preprocessing, and batch creation
- `models`: Neural network architecture and initialization, of optimizer and loss functions
- `plots`: Visualization utilities for training metrics
- `utils`: Helper functions for training and evaluation

**Action Required**: 
1. Update the path in the sys.path.append() line to match your nntab installation location
2. Run the cells to import the nntab package and its modules

In [3]:
import sys
sys.path.append("/jupyterhubenc/e106496/nn_package/version4/work_dir/nn_tab")

In [4]:
!ls -1

config.json
example_training_notebook.ipynb
logs
main.py
models
nn_tab
parameters
README.md


In [5]:
import nn_tab.datasets as datasets
import nn_tab.models as models
import nn_tab.plots as plots
import nn_tab.utils as utils

In [6]:
from nn_tab.datasets import dataset, preprocess, dataloading_helpers
from nn_tab.models import model_initialise, model
from nn_tab.utils import utils
from nn_tab.plots import plots

### loading the dataset 

## Step 3: Configure Dataset Paths and Load Training Data

**Purpose**: Specify the locations of your training data and feature configuration files, then load and prepare the data for training.

**Required Files**:
1. **Data File** (`data_path`): Your training dataset in CSV or Parquet format
   - Must contain your feature columns and target variable
   - Supported formats: `.csv`, `.parquet`

2. **Feature Configuration File** (`feat_path`): CSV file listing all features to use
   - Should contain column names that match your data file
   - Example path: `/path/to/your/features.csv`

**Action Required**: 
1. **Update the data_path** variable to point to your training dataset
2. **Update the feat_path** variable to point to your feature configuration file
3. Run the cell to set these paths

In [7]:
# data_path = "/ads_storage/e151283/SSFT/train_split1.parquet"
# feat_path = "/ads_storage/e106496/Research/nn_for_di/datasets/di_all_features.csv"

In [8]:
data_path = "/ads_storage/e151283/SSFT/train_split1.parquet"
feat_path = "/ads_storage/e151283/SSFT/feature_names_DIpro.pkl"

## Step 4: Set Training Hyperparameters

**Purpose**: Configure the key parameters that control the training process and model behavior.

**Hyperparameters Explained**:
- **exp_name**: Name for this experiment (used for saving models and logs)
  - Example: "fraud_detection_v1", "test_run_20240122"
  
- **epochs**: Number of complete passes through the training dataset
  - More epochs = longer training but potentially better performance
  - Typical range: 10-100 (start with 10 for testing)
  
- **batch_size**: Number of samples processed before updating model weights
  - Larger batch = faster training but more memory
  - Typical values: 16, 32, 64, 128, 256
  
- **num_workers**: Number of parallel data loading processes
  - Set to 0 for debugging, 2-4 for production
  - Higher values speed up data loading but use more memory
  
- **device**: Computing device for training (GPU or CPU)
  - Automatically detects and uses GPU if available
  - GPU significantly speeds up training

**Action Required**: 
1. Modify these parameters according to your needs
2. Run the cell to set the hyperparameters

In [9]:
exp_name = "test_di"
epochs = 10
batch_size = 32
num_workers= 0
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [10]:
loss_function = "CrossEntropyLoss"
model_name = "fraudmodel_5layer"
training_method = "standard_multiclass"

In [11]:
device

device(type='cpu')

In [12]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.enabled)
print(torch.cuda.is_available())

RecursionError: maximum recursion depth exceeded while calling a Python object

## Step 5: Load and Prepare Training Data

**Purpose**: Read the dataset, preprocess features, create data loaders, and compute class weights for handling imbalanced data.

**What this cell does**:
1. **Reads the data** from your specified paths
2. **Preprocesses features** according to the feature configuration
3. **Splits data** into training and validation sets
4. **Creates data loaders** for batch processing during training
5. **Calculates class weights** to handle imbalanced fraud detection datasets

**Function Parameters**:
- `feat_path`: Path to feature configuration file
- `"approved_dll_fraud_sw_latest"`: Target column name in your dataset
- `data_path`: Path to your training data
- `batch_size`: Batch size for training
- `num_workers`: Number of data loading workers

**Outputs**:
- `train_loader`: PyTorch DataLoader for training data
- `val_loader`: PyTorch DataLoader for validation data
- `class_weights`: Weights for each class (handles imbalance)
- `feat_list`: List of feature names used in the model

**Action Required**: 
1. Ensure the target column name matches your dataset
2. Run the cell to load and prepare the data
3. Check the output to verify data loaded successfully

In [ ]:
train_loader, val_loader, class_weights, feat_list = dataset.read_train_data(feat_path, "approved_dll_fraud_sw_latest", data_path, batch_size, num_workers)

## Step 6: Verify Data Loaders (Optional)

**Purpose**: Check the number of batches in training and validation data loaders to ensure data was loaded correctly.

**What to expect**:
- Output shows: (number_of_training_batches, number_of_validation_batches)
- Example: (1000, 250) means 1000 training batches and 250 validation batches

**Action Required**: Run the cell to verify data loaders are properly configured

In [ ]:
len(train_loader), len(val_loader)

## Step 7: Initialize the Neural Network Model

**Purpose**: Create and configure the neural network architecture with appropriate layers, loss function, optimizer, and learning rate scheduler.

**Neural Network Architecture**:
- **Input Layer**: Size = number of features in your dataset (automatically set from `feat_list`)
- **Hidden Layers**: 
  - Layer 1: 512 neurons
  - Layer 2: 128 neurons
  - Layer 3: 64 neurons
  - Layer 4: 4 neurons
- **Output Layer**: 2 neurons (for binary classification: fraud/not fraud)

**Model Components Initialized**:
1. **model**: The neural network architecture
2. **criterion**: Loss function (uses class weights for imbalanced data)
3. **optimizer**: Algorithm for updating model weights (e.g., Adam)
4. **scheduler**: Learning rate adjustment strategy during training

**Customization Options**:
You can modify the hidden layer sizes (h1, h2, h3, h4) based on:
- Dataset complexity
- Number of features
- Available computational resources
- Typical starting point: Larger initial layers, gradually decreasing

**Action Required**: 
1. Adjust hidden layer sizes if needed (optional)
2. Run the cell to initialize the model
3. Check output for model summary and parameter count

In [ ]:
model, criterion, optimizer, scheduler = model_initialise.get_model(device,
                                                                    model_name=model_name,
                                                                    input_dim=len(feat_list),
                                                                    class_weights=class_weights,
                                                                    hidden_layers=[512, 128, 64, 4],
                                                                    output=2,
                                                                    loss_function=loss_function,
                                                                    focal_loss_config={"gamma": 2.0})

In [ ]:
model

In [ ]:
criterion

In [ ]:
optimizer

In [ ]:
scheduler

## Step 8: Train the Neural Network Model

**Purpose**: Execute the complete training loop for the specified number of epochs, including training on the training set and validation on the validation set.

**What happens during training**:
1. **Forward Pass**: Data flows through the network to generate predictions
2. **Loss Calculation**: Compares predictions with actual labels
3. **Backward Pass**: Computes gradients for all model parameters
4. **Weight Update**: Optimizer adjusts model weights based on gradients
5. **Validation**: Model performance evaluated on validation set after each epoch
6. **Checkpointing**: Best model saved based on validation performance

**Expected Output**:
- Epoch-by-epoch training progress
- Training loss and validation loss for each epoch
- Validation metrics (accuracy, precision, recall, F1-score, AUC)
- Model checkpoint saves with experiment name

**Training Artifacts**:
The training process will create:
- Model checkpoints (saved with `exp_name`)
- Training logs with loss curves
- Validation metrics history

**Monitoring Training**:
- Training loss should decrease over epochs
- If loss increases or plateaus early, consider adjusting learning rate or batch size
- Monitor for overfitting (training loss much lower than validation loss)

**Action Required**: 
1. Run the cell to start training
2. Monitor the output for training progress
3. Wait for all epochs to complete (this may take several minutes to hours depending on dataset size and hardware)

In [ ]:
print("starting the training!!")
utils.train_model(model,
                  epochs,
                  optimizer,
                  scheduler,
                  train_loader,
                  val_loader,
                  criterion,
                  device,
                  exp_name,
                  model_name=model_name,
                  training_method=training_method)

## Troubleshooting Guide

### Common Issues and Solutions:

**1. CUDA Out of Memory Error**
- **Solution**: Reduce `batch_size` (try 16 or 32)
- Or reduce hidden layer sizes
- Or set `device = torch.device("cpu")` to use CPU

**2. FileNotFoundError**
- **Solution**: Verify `data_path` and `feat_path` are correct
- Use absolute paths instead of relative paths
- Check file permissions

**3. Import Error for nntab**
- **Solution**: Verify `sys.path.append()` points to correct location
- Ensure nntab package is properly installed
- Check that all `__init__.py` files exist in nntab directories

**4. Training Loss Not Decreasing**
- **Solution**: Check learning rate (may need adjustment)
- Verify data is properly normalized/preprocessed
- Try training for more epochs
- Check for data leakage or errors

**5. Validation Performance Poor**
- **Solution**: Model may be underfitting - try larger network or more epochs
- Or overfitting - add regularization or reduce model size
- Check class balance and adjust `class_weights`

**6. Slow Training Speed**
- **Solution**: Increase `num_workers` for faster data loading
- Use GPU if available
- Increase `batch_size` if memory allows

### Next Steps After Training:
1. Check saved model checkpoints in your experiment folder
2. Analyze training curves and validation metrics
3. Load best model for inference on test data
4. Fine-tune hyperparameters based on results